# 듀오링고 첫 레슨 — 가상 데이터 생성

신규 사용자와 레슨 이벤트를 만들어 `users.csv`, `event_logs.csv`로 저장하는 노트북.

- 신규 사용자 유입: 2026-07-01 ~ 2026-09-30 (가상)
- 이벤트 관측 종료: 2026-10-09
- 목표 퍼널: 첫 레슨 시작 10,000명 → 첫 답안 9,200명 → 절반 도달 7,800명 → 첫 레슨 완료 6,500명 → 두 번째 레슨 시작 2,730명
- 설계 가설: H1 구독 안내, H2 말하기 건너뛰기, H3 에너지, H4 힌트, H5 일일 목표

> **주의:** 생성 확률과 행동은 실측치가 아닌 프로젝트용 가정. 실제 듀오링고 데이터가 아님.


## 1. 라이브러리 및 데이터 경로

기존 호출 코드와 PC 경로 설정 유지.

In [1]:
from pathlib import Path
import csv
import json
import random

# Counter: 이벤트별 건수 집계
from collections import Counter
# datetime, timedelta: 날짜·시간 기록과 간격 계산
from datetime import datetime, timedelta

In [2]:
# 1. 프로젝트 최상위 경로: 내 PC의 GitHub 저장소 위치
# 폴더 위치가 다를경우 아래 ROOT 문자열만 수정
ROOT = Path(r"C:\Users\joon5\Documents\github\duolingo-funnel-analysis")

# 2. 생성 데이터와 전처리 데이터의 폴더 분리
DATA_RAW_DIR = ROOT / "data" / "raw"
DATA_PROCESSED_DIR = ROOT / "data" / "processed"

# 3. 생성할 CSV 이름
USER_PROFILE_FILE = "users.csv"
EVENT_LOG_FILE = "event_logs.csv"

# 4. 실제 저장할 전체 파일 경로
USER_PROFILE_PATH = DATA_RAW_DIR / USER_PROFILE_FILE
EVENT_LOG_PATH = DATA_RAW_DIR / EVENT_LOG_FILE

# 다른 폴더에 잘못 저장되는 상황 방지용 프로젝트 경로 확인
if not ROOT.is_dir():
    raise FileNotFoundError("프로젝트 폴더가 없습니다. 위 ROOT 경로를 본인 PC에 맞게 수정하세요.")

# 저장 폴더가 없는 경우 생성, 이미 존재하면 유지
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("프로젝트 폴더 :", ROOT)
print("사용자 저장 경로:", USER_PROFILE_PATH)
print("이벤트 저장 경로:", EVENT_LOG_PATH)
print("전처리 폴더   :", DATA_PROCESSED_DIR)


프로젝트 폴더 : C:\Users\joon5\Documents\github\duolingo-funnel-analysis
사용자 저장 경로: C:\Users\joon5\Documents\github\duolingo-funnel-analysis\data\raw\users.csv
이벤트 저장 경로: C:\Users\joon5\Documents\github\duolingo-funnel-analysis\data\raw\event_logs.csv
전처리 폴더   : C:\Users\joon5\Documents\github\duolingo-funnel-analysis\data\processed


## 2. 생성 규칙 설정

수정할 수 있는 기본값을 한곳에 정리한 셀.

- `STAGE_WEIGHTS`: 각 사용자가 **마지막으로 도달하는 단계**의 확률. 누적 전환율과 구분.
- H1~H3의 확률: 가설별 집단 차이를 **가상으로 심어 둔 값**.
- 일일 학습 목표(3·10·15·30분)와 문제 수(10~16개): 별도로 추첨.


In [3]:
# 기간: 7~9월 유입, 10월 9일까지 이벤트 관측
FIRST_SEEN_START = datetime(2026, 7, 1)
FIRST_SEEN_END = datetime(2026, 10, 1)  # 종료일 제외
LAST_LESSON_START = datetime(2026, 9, 30, 23, 59, 59)
OBSERVATION_END = datetime(2026, 10, 9, 23, 59, 59)

# users.csv 저장 컬럼 순서
USER_COLUMNS = (
    "user_id", "first_seen_at", "course_type", "starting_point",
    "self_assessed_level", "daily_goal_min",
)

# event_logs.csv 저장 컬럼 순서
# 이벤트마다 다른 정보는 event_properties에 JSON 문자열로 저장
EVENT_COLUMNS = (
    "event_id", "user_id", "event_time", "event_name",
    "lesson_number", "lesson_attempt_id", "event_properties",
)

# 한 레슨은 10~16문제, 15문제가 가장 자주 나오도록 설정
QUESTION_COUNTS = (10, 11, 12, 13, 14, 15, 16)
QUESTION_COUNT_WEIGHTS = (2, 3, 5, 10, 20, 40, 20)

# 영어 레슨 문제 유형별 출제 비중: 시뮬레이션용 가정
QUESTION_TYPES = ("word_selection", "sentence_completion", "listening", "speaking")
QUESTION_TYPE_WEIGHTS = (30, 30, 20, 20)

# H5: 실제 앱에서 관찰한 일일 학습 목표 선택지
DAILY_GOALS = (3, 10, 15, 30)
DAILY_GOAL_WEIGHTS = (25, 40, 25, 10)

# 자기평가 영어 수준(1~5)과 임의 분포
LEVELS = (1, 2, 3, 4, 5)
LEVEL_WEIGHTS = (25, 30, 25, 15, 5)

# 각 사용자의 최종 도달 단계 (누적 비율이 아님)
# 1: 시작만, 2: 답안 제출까지, 3: 절반까지, 4: 첫 레슨 완료, 5: 두 번째 레슨 시작
STAGES = (1, 2, 3, 4, 5)
STAGE_WEIGHTS = (8.0, 14.0, 13.0, 37.7, 27.3)

# H2: 첫 레슨 완료자/미완료자에게 말하기 건너뛰기를 부여하는 확률
# 두 집단을 다르게 설정하여 건너뛰기 경험군의 낮은 완료율을 가상으로 표현
P_SKIP_COMPLETE = 0.127
P_SKIP_INCOMPLETE = 0.193

# H1: 두 번째 레슨 시작자/미시작자의 구독 안내 노출 확률
# 완료자 중 약 60% 노출, 노출군의 낮은 후속 시작률을 가상으로 표현
P_OFFER_SECOND = 0.557
P_OFFER_NO_SECOND = 0.631

# H3: 두 번째 레슨 시작자/미시작자의 낮은 에너지(0~5) 확률
# 완료자 중 약 25% 부족군, 부족군의 낮은 후속 시작률을 가상으로 표현
P_LOW_ENERGY_SECOND = 0.179
P_LOW_ENERGY_NO_SECOND = 0.302


## 3. 공통 보조 함수

중복되는 작은 작업만 함수로 구성.

- `choose()`: 후보 중 가중치에 따라 무작위 선택
- `stamp()`: 날짜·시간을 CSV용 문자열로 변환
- `add_event()`: 이벤트 한 건을 같은 구조로 기록


In [4]:
# 주어진 후보 중 가중치에 따라 하나 선택
# 예: choose(rng, (3, 10), (25, 75)) → 3분 약 25%, 10분 약 75%
def choose(rng, options, weights):
    return rng.choices(options, weights=weights, k=1)[0]


# datetime 자료를 '2026-07-01 10:00:00' 문자열로 변환
def stamp(value):
    return value.strftime("%Y-%m-%d %H:%M:%S")


# 이벤트를 메모리 목록에 한 건씩 추가
# events 안의 순서: 시간, 기록 순서, 사용자 ID, 이벤트명, 레슨 번호, 시도 ID, 속성 JSON
def add_event(events, when, user_id, name, lesson_number, attempt_id, properties):
    # JSON 형태의 추가 속성을 CSV 한 칸에 들어갈 문자열로 변환
    properties_text = json.dumps(properties, ensure_ascii=False)
    events.append((
        when, len(events), user_id, name,
        lesson_number, attempt_id, properties_text,
    ))


## 4. 사용자 한 명의 행동 생성

한 사용자의 **프로필 → 레슨 시작 → 문제 풀이 → 완료 또는 중단 → 다음 레슨** 순서로 생성.

1. 사용자 특성 및 마지막 퍼널 단계 추첨
2. 레슨 문제 수와 건너뛰기·힌트·에너지·구독 안내 상태 결정
3. 문제 조회 → 답안 제출 → 진행률 50% → 완료/중단 이벤트 기록
4. 첫 레슨 완료자 일부에게 후속 레슨 시작 이벤트 기록

진행률은 `정답으로 완료한 필수 문제 수 ÷ 전체 필수 문제 수 × 100`. 오답과 건너뛰기에서는 증가하지 않음.


In [5]:
def generate_user(user_number, rng, events):
    # ---------- 1) 사용자 정보 ----------
    # U00001처럼 사용자 고유 번호 생성
    user_id = f"U{user_number:05d}"

    # 7월 1일부터 9월 30일까지 무작위 최초 이용 시각 설정
    total_seconds = int((FIRST_SEEN_END - FIRST_SEEN_START).total_seconds())
    first_seen_at = FIRST_SEEN_START + timedelta(seconds=rng.randrange(total_seconds))

    # 최초 이용 후 1~60분 사이 첫 레슨 시작, 9월 30일 이후 시작 방지
    first_lesson_start = first_seen_at + timedelta(seconds=rng.randint(60, 3600))
    first_lesson_start = min(first_lesson_start, LAST_LESSON_START)

    # 자기평가 수준이 높을수록 중급 영어를 선택할 확률 증가
    level = choose(rng, LEVELS, LEVEL_WEIGHTS)
    intermediate_chance = {1: 0.07, 2: 0.17, 3: 0.35, 4: 0.70, 5: 0.90}[level]
    intermediate = rng.random() < intermediate_chance
    course_type = "intermediate_english" if intermediate else "beginner_english"

    # 중급 사용자에게 수준 평가(placement_test) 선택 확률을 높게 설정
    placement_chance = 0.45 if intermediate else 0.17
    starting_point = "placement_test" if rng.random() < placement_chance else "start_basics"

    # H5: 일일 학습 목표 시간 추첨 (문제 수에는 영향 없음)
    daily_goal_min = choose(rng, DAILY_GOALS, DAILY_GOAL_WEIGHTS)
    user_row = [user_id, stamp(first_seen_at), course_type, starting_point, level, daily_goal_min]

    # ---------- 2) 최종 도달 단계와 레슨 구성 ----------
    # 예: 3단계 → 50%에는 도달하지만 첫 레슨을 완료하지 않는 사용자
    last_stage = choose(rng, STAGES, STAGE_WEIGHTS)
    completed = last_stage >= 4
    second_started = last_stage == 5

    total_questions = choose(rng, QUESTION_COUNTS, QUESTION_COUNT_WEIGHTS)
    # //는 몫 계산: 15문제일 때 8문제 완료부터 진행률 50% 이상
    halfway = (total_questions + 1) // 2
    attempt_id = f"A{user_number:07d}"

    # 최종 단계에 따라 맞혀서 완료할 문제 개수 결정
    if last_stage == 1:
        correct_target = 0
    elif last_stage == 2:
        correct_target = rng.randint(1, halfway - 1)
    elif last_stage == 3:
        correct_target = rng.randint(halfway, total_questions - 1)
    else:
        correct_target = total_questions

    # ---------- 3) 가설 관련 상태 결정 ----------
    # H2: 미완료자에게 말하기 문제 건너뛰기가 더 자주 생기도록 설정
    skip_chance = P_SKIP_COMPLETE if completed else P_SKIP_INCOMPLETE
    speaking_skip = rng.random() < skip_chance

    # 실제로 화면에 표시되는 문제 안에서 건너뛸 문제 번호 선택
    # 미완료자는 마지막에 '보기만 한 문제' 하나가 추가되므로 +1
    last_viewed_question = correct_target if completed else correct_target + 1
    skipped_question = rng.randint(1, last_viewed_question) if speaking_skip else 0

    # H3: 완료자에게만 에너지 잔량 생성
    low_energy = False
    energy_remaining = None
    if completed:
        low_chance = P_LOW_ENERGY_SECOND if second_started else P_LOW_ENERGY_NO_SECOND
        low_energy = rng.random() < low_chance
        if low_energy:
            energy_remaining = rng.randint(0, 5)
        else:
            energy_remaining = rng.randint(6, 25)

    # 사용자 수준이 낮을수록 오답 확률 증가 (프로젝트 가정)
    error_chance = {1: 0.29, 2: 0.24, 3: 0.19, 4: 0.14, 5: 0.09}[level]
    if low_energy:
        error_chance += 0.14

    # ---------- 4) 첫 레슨 문제 풀이 ----------
    current_time = first_lesson_start
    add_event(events, current_time, user_id, "lesson_started", 1, attempt_id,
              {"total_questions": total_questions})

    correct_done = 0
    hint_checked = False   # 힌트 사용 여부 추첨을 이미 했는지 구분
    hint_will_use = False  # 추첨 결과: 힌트를 사용할지 구분

    for question_number in range(1, correct_target + 1):
        # 건너뛰기 대상은 말하기 문제로 고정, 나머지 문제 유형은 무작위
        if question_number == skipped_question:
            question_type = "speaking"
        else:
            question_type = choose(rng, QUESTION_TYPES, QUESTION_TYPE_WEIGHTS)

        attempt = 1
        current_time += timedelta(seconds=rng.randint(3, 10))
        add_event(events, current_time, user_id, "question_viewed", 1, attempt_id,
                  {"question_number": question_number, "attempt_number": attempt,
                   "question_type": question_type})

        # H2: 말하기 건너뛰기 → 같은 필수 문제 번호에 대체 유형 노출
        if question_number == skipped_question:
            current_time += timedelta(seconds=rng.randint(2, 12))
            add_event(events, current_time, user_id, "question_skipped", 1, attempt_id,
                      {"question_number": question_number, "attempt_number": attempt,
                       "question_type": "speaking", "skip_reason": "cannot_speak_now"})

            question_type = rng.choice(("word_selection", "sentence_completion", "listening"))
            attempt += 1
            current_time += timedelta(seconds=rng.randint(3, 10))
            add_event(events, current_time, user_id, "question_viewed", 1, attempt_id,
                      {"question_number": question_number, "attempt_number": attempt,
                       "question_type": question_type})

        # 일부 문제에서 오답 → 재시도. 오답 자체는 진행률에 반영하지 않음
        if rng.random() < error_chance:
            current_time += timedelta(seconds=rng.randint(8, 42))
            add_event(events, current_time, user_id, "answer_submitted", 1, attempt_id,
                      {"question_number": question_number, "is_correct": False,
                       "progress_percent": round(correct_done / total_questions * 100, 2)})

            attempt += 1
            current_time += timedelta(seconds=rng.randint(3, 10))
            add_event(events, current_time, user_id, "question_viewed", 1, attempt_id,
                      {"question_number": question_number, "attempt_number": attempt,
                       "question_type": question_type})

            # H4: 첫 오답에서만 힌트 이용 여부 30% 추첨
            if not hint_checked:
                hint_will_use = rng.random() < 0.30
                hint_checked = True

            # 이번 생성 규칙에서는 사용자당 힌트 사용을 최대 한 번 기록
            if hint_will_use:
                current_time += timedelta(seconds=rng.randint(2, 8))
                add_event(events, current_time, user_id, "hint_used", 1, attempt_id,
                          {"question_number": question_number, "attempt_number": attempt})
                hint_will_use = False

        # 정답 제출 → 완료 문제 수 증가 → 진행률 갱신
        current_time += timedelta(seconds=rng.randint(8, 42))
        correct_done += 1
        progress = round(correct_done / total_questions * 100, 2)
        add_event(events, current_time, user_id, "answer_submitted", 1, attempt_id,
                  {"question_number": question_number, "is_correct": True,
                   "progress_percent": progress})

        # 절반 이상에 처음 도달한 순간 이벤트 한 건 추가
        if correct_done == halfway:
            current_time += timedelta(seconds=1)
            add_event(events, current_time, user_id, "lesson_halfway_reached", 1, attempt_id, {})

    # ---------- 5) 중도 이탈 또는 첫 레슨 완료 ----------
    if not completed:
        # 마지막 문제를 화면에서 본 후 답안을 제출하지 않고 이탈
        question_number = correct_target + 1
        question_type = "speaking" if question_number == skipped_question else choose(
            rng, QUESTION_TYPES, QUESTION_TYPE_WEIGHTS
        )
        current_time += timedelta(seconds=rng.randint(3, 10))
        add_event(events, current_time, user_id, "question_viewed", 1, attempt_id,
                  {"question_number": question_number, "attempt_number": 1,
                   "question_type": question_type})

        # 마지막 문제의 건너뛰기 선택 후 곧바로 종료하는 경우
        if question_number == skipped_question:
            current_time += timedelta(seconds=rng.randint(2, 12))
            add_event(events, current_time, user_id, "question_skipped", 1, attempt_id,
                      {"question_number": question_number, "attempt_number": 1,
                       "question_type": "speaking", "skip_reason": "cannot_speak_now"})

        return user_row

    # 모든 필수 문제 완료 시 lesson_completed 생성
    current_time += timedelta(seconds=rng.randint(1, 5))
    completion_time = current_time
    add_event(events, completion_time, user_id, "lesson_completed", 1, attempt_id,
              {"energy_remaining": energy_remaining})

    # H1: 첫 레슨 완료 후 일부 사용자에게 구독 안내 화면 노출
    offer_chance = P_OFFER_SECOND if second_started else P_OFFER_NO_SECOND
    if rng.random() < offer_chance:
        current_time += timedelta(seconds=rng.randint(2, 15))
        add_event(events, current_time, user_id, "subscription_offer_viewed", 1, None, {})

    # 5단계 도달자만 7일 이내 두 번째 레슨 시작 이벤트 생성
    if second_started:
        second_time = completion_time + timedelta(seconds=rng.randint(3600, 6 * 24 * 3600))
        add_event(events, second_time, user_id, "lesson_started", 2,
                  f"A{user_number + 10000:07d}",
                  {"total_questions": choose(rng, QUESTION_COUNTS, QUESTION_COUNT_WEIGHTS)})

    return user_row


## 5. CSV 파일 생성

사용자 1명씩 생성하면서 `users.csv` 저장, 모든 이벤트를 날짜·시간순으로 정렬한 뒤 `event_logs.csv` 저장.

확인은 중복 출력 없이 **마지막 7번 셀**에서 한 번만 진행.


In [6]:
def generate(output_dir, n_users, seed, overwrite):
    users_path = output_dir / "users.csv"
    events_path = output_dir / "event_logs.csv"

    # 기존 CSV 보호: 덮어쓰기를 허용하지 않았는데 파일이 이미 있으면 중단
    if not overwrite and (users_path.exists() or events_path.exists()):
        raise FileExistsError("CSV 파일이 이미 존재함. 재생성 시 OVERWRITE_EXISTING = True로 변경")

    # 같은 시드(seed)를 사용하면 동일한 무작위 결과 재현 가능
    rng = random.Random(seed)
    events = []

    # 사용자 한 명 생성 → users.csv 한 행 저장
    # newline='' 옵션은 CSV 저장 시 불필요한 빈 행 생성 방지
    with users_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(USER_COLUMNS)

        for user_number in range(1, n_users + 1):
            user_row = generate_user(user_number, rng, events)
            writer.writerow(user_row)

    # 튜플 맨 앞이 시각이므로 전체 이벤트를 발생 시각순으로 정렬
    # 같은 초에 발생한 이벤트는 두 번째 값(입력 순서)으로 정렬
    events.sort()

    # event_id를 순서대로 붙이고 이벤트 CSV 저장
    with events_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(EVENT_COLUMNS)

        for event_number, event in enumerate(events, start=1):
            when, order, user_id, name, lesson_number, attempt_id, properties = event
            writer.writerow([
                f"E{event_number:09d}", user_id, stamp(when), name,
                lesson_number, attempt_id or "", properties,
            ])

    print("CSV 생성 완료")


## 6. 생성 실행

먼저 `N_USERS = 100`으로 테스트한 뒤 정상 작동 시 `10_000`으로 변경.

- `RANDOM_SEED`: 무작위 결과 재현을 위한 숫자
- `OVERWRITE_EXISTING`: 기존 파일 덮어쓰기 여부 (처음에는 `False`)

> 100명 테스트 후 10,000명을 생성할 때는 기존 CSV가 있으므로 `OVERWRITE_EXISTING = True`로 변경 필요.


In [7]:
# 생성할 사용자 수: 처음에는 100명 테스트, 최종 실행은 10_000명
N_USERS = 10_000

# 랜덤 시드 고정: 같은 설정이라면 같은 결과 재현
RANDOM_SEED = 20261111

# 기존 파일 덮어쓰기 여부: 재생성 시에만 True
OVERWRITE_EXISTING = True

# ROOT/data/raw에 users.csv와 event_logs.csv 저장
generate(DATA_RAW_DIR, N_USERS, RANDOM_SEED, OVERWRITE_EXISTING)


CSV 생성 완료


## 7. 결과 확인 (마지막에 한 번)

두 CSV를 읽고 **사용자 수, 이벤트 수, 5단계 누적 퍼널, 완료율, 후속 시작률**만 확인.

H1~H5 집단별 상세 비교와 통계 검정은 별도 분석 노트북에서 진행.


In [8]:
# 저장된 사용자 CSV 호출
with USER_PROFILE_PATH.open(encoding="utf-8", newline="") as file:
    users = list(csv.DictReader(file))

# 각 퍼널 단계에 도달한 사용자 ID 저장용 집합
# set 자료형 사용 이유: 이벤트가 여러 건이어도 사용자 1명만 집계
funnel_users = {1: set(), 2: set(), 3: set(), 4: set(), 5: set()}
event_count = 0
event_types = Counter()

# 저장된 이벤트 CSV 호출 및 첫 레슨/두 번째 레슨 구분
with EVENT_LOG_PATH.open(encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        event_count += 1
        event_name = row["event_name"]
        user_id = row["user_id"]
        lesson_number = int(row["lesson_number"])
        event_types[event_name] += 1

        if lesson_number == 1:
            if event_name == "lesson_started":
                funnel_users[1].add(user_id)
            elif event_name == "answer_submitted":
                funnel_users[2].add(user_id)
            elif event_name == "lesson_halfway_reached":
                funnel_users[3].add(user_id)
            elif event_name == "lesson_completed":
                funnel_users[4].add(user_id)
        elif lesson_number == 2 and event_name == "lesson_started":
            funnel_users[5].add(user_id)

# 생성 규모와 퍼널 순서의 기본 확인
assert len(users) == N_USERS, "사용자 수 확인 필요"
assert len({user["user_id"] for user in users}) == N_USERS, "사용자 ID 중복 확인 필요"
for stage in range(2, 6):
    assert funnel_users[stage].issubset(funnel_users[stage - 1]), "퍼널 단계 순서 확인 필요"

# 5단계 누적 퍼널 출력: 다음 단계 도달자 / 직전 단계 도달자
print(f"사용자 {len(users):,}명 / 이벤트 {event_count:,}건 / 이벤트 종류 {len(event_types)}개")
print("단계 | 도달 사용자 | 직전 단계 전환율")
for stage in range(1, 6):
    current = len(funnel_users[stage])
    previous = len(funnel_users[stage - 1]) if stage > 1 else len(users)
    print(f"{stage}단계 | {current:,}명 | {current / previous * 100:.2f}%")

# 첫 레슨 완료율의 분모는 첫 레슨 시작자, 두 번째 시작률의 분모는 첫 레슨 완료자
first_rate = len(funnel_users[4]) / len(funnel_users[1]) * 100
second_rate = len(funnel_users[5]) / len(funnel_users[4]) * 100
print(f"첫 레슨 완료율: {first_rate:.2f}%")
print(f"두 번째 레슨 시작률: {second_rate:.2f}%")


사용자 10,000명 / 이벤트 320,705건 / 이벤트 종류 8개
단계 | 도달 사용자 | 직전 단계 전환율
1단계 | 10,000명 | 100.00%
2단계 | 9,226명 | 92.26%
3단계 | 7,824명 | 84.80%
4단계 | 6,485명 | 82.89%
5단계 | 2,815명 | 43.41%
첫 레슨 완료율: 64.85%
두 번째 레슨 시작률: 43.41%
